# Module-Level Connectivity — Confirmatory Analysis

---

### Overview

This notebook reproduces and explains every step of the **module-level confirmatory analysis** pipeline for depression subtyping. It tests whether anatomical modules derived from a previous modularisation (LRG community detection) show differential connectivity across:

- **Depression vs. Control** (main effect of diagnosis)
- **Cluster 0 vs. Control** (cluster-specific effect)
- **Cluster 1 vs. Control** (cluster-specific effect)
- **Cluster 0 vs. Cluster 1** (inter-cluster difference)

The cluster of interest is always `sfc_external_cluster` — the subject-level cluster assignment from the exploratory SFC-external clustering step (`module_clustering_main.py`).

---

### What this notebook does — step by step

| Step | Description |
|------|-------------|
| 1 | **Imports & configuration** — load libraries, set all paths, load the module features CSV, community labels, and column-name mapping |
| 2 | **Output capture helpers** — define `Tee` and `capture_stdout_to_log` to mirror all stdout to a plain-text log file |
| 3 | **R environment initialisation** — call `setup_r_environment()` to activate `rpy2` and verify `quantreg`/`boot` packages |
| 4 | **Quantile-regression pipeline** — call `run_module_quantile_regression_pipeline()` which iterates over all modules × directions × connectivity types, fits three quantile regression models per module in R, collects p-values, and applies FDR-BH correction per (connectivity, direction) family |
| 5 | **Violin plots** — produced automatically inside the pipeline for each connectivity type and direction, annotated with FDR-corrected significance brackets |
| 6 | **Brainmap visualisation** — call `plot_cluster_feature_brainmaps()` to render Cluster 0 and Cluster 1 median module profiles and their difference as NIfTI/SVG brain images, restricted to modules that pass the FDR threshold |
| 7 | **Output summary** — verify and list all saved files |

---

### Utility module — `module_clustering_confirmatory_utils.py`

The heavy lifting is delegated to this utility module, which provides:

- **`setup_r_environment()`** — initialise rpy2 and import required R packages.
- **`run_quantile_regression(combined_df, ...)`** — write data to a temporary CSV, call R's `quantreg::rq` (median, τ = 0.5) with bootstrapped SEs (`se = "boot"`), run three models (depression vs control; clusters vs control; cluster 0 vs cluster 1), extract p-values, and return a dict keyed by comparison name.
- **`run_module_quantile_regression_pipeline(...)`** — orchestrate the per-module loop, collect all raw p-values, run Benjamini–Hochberg FDR correction per (connectivity type, direction) family, build the `significance_map`, and call `plot_module_violin_across_clusters` for each connectivity type.
- **`plot_module_violin_across_clusters(final_df, feature_df, ...)`** — for each connectivity type and direction create one figure with one violin panel per module showing Control / Depression / Cluster 0 / Cluster 1 distributions (min-max scaled to [0, 1]) annotated with FDR-corrected significance brackets.
- **`plot_cluster_feature_brainmaps(final_df, feature_df, atlas_img_path, community_labels, ...)`** — compute per-cluster median module profiles, project them onto a brain atlas, and save side-by-side (Cluster 0 | Cluster 1) and difference (Cluster 0 − Cluster 1) maps.
- **`apply_multiple_testing_correction(...)`** — Benjamini–Hochberg FDR (or Bonferroni) correction with optional log output.

---

### Expected inputs

| Variable | Type | Description |
|----------|------|-------------|
| `DATA` | `pd.DataFrame` | Per-subject module connectivity features + all covariates. Produced by `module_clustering_main.py`. Must contain `eid`, `depression_status` (0/1), `sfc_external_cluster`, and feature columns `<module>_{dir}_{conn}` (e.g. `X9_0_external_functional`). |
| `colname_map` | `dict` | Optional. Maps original feature names → R-safe sanitised column names. Loaded from a two-column CSV (`original`, `sanitized`). |
| `CM` | `np.ndarray` | Community label array (one integer per ROI, matching atlas ROI ordering) loaded from `CM.txt`. |
| `ATLAS_FILE` | `str` | Path to the Schaefer 1000 + TianS4 parcellation NIfTI used for brainmap projections. |

Key configuration constants (set in the configuration cell):

| Constant | Description |
|----------|-------------|
| `CONNECTIVITY_TYPES` | Tuple of connectivity modalities: `('functional', 'structural', 'sfc')` |
| `DIRECTION_TYPES` | Tuple of directions: `('internal', 'external')` |
| `ICD_10_COVARIATES` | ICD-10 comorbidity columns included as covariates (hypertension, substance history, anxiety) |
| `CLUSTER_COL` | Cluster column name used for grouping: `'sfc_external_cluster'` |
| `QUANTILE_REGRESSION_BOOTSTRAP_R` | Number of bootstrap iterations for R SEs (default 1000; increase for final publication runs) |
| `fMRI_MOTION_METRIC` / `dMRI_MOTION_METRIC` | Head-motion covariate column names |

---

### Expected outputs

| Output | Location | Description |
|--------|----------|-------------|
| Full stdout + R output log | `DEPRESSION_DIR/module_connectivity_output_sfc_external.txt` | Complete run transcript |
| FDR correction logs | `DEPRESSION_DIR/modular_{conn}_{dir}_connectivity_FDR_sfc_external.txt` | Per-family raw + corrected p-values |
| Violin plot SVGs | `PLOTS_DIR/` | One figure per (connectivity type × direction) with per-module violin panels annotated with `*`/`**`/`***`/`ns` |
| Brainmap SVGs | `FIGURES_DIR/{conn}_con/` | Cluster 0, Cluster 1, and difference brain maps for significant modules |
| NIfTI masks | `FIGURES_DIR/nifti_masks/` | Optional volumetric masks for each map |

---

### Runtime notes

- Requires a working **R installation** with `quantreg` and `boot` packages accessible via `rpy2`.
- Each module regression writes a temporary CSV to `/tmp/`; avoid running multiple pipeline instances in parallel.
- Bootstrap iteration count (`QUANTILE_REGRESSION_BOOTSTRAP_R = 1000`) is moderate; increase to 10 000 for publication-grade estimates.
- The notebook mirrors the exact call sequence of `module_clustering_confirmatory_main.py`; if you need to re-run specific steps only, skip to the relevant cell.

## Step 1 — Imports and configuration

Load all required Python libraries, define paths, load the main data CSV, community labels, and column-name mapping. All constants used throughout the notebook are set here — adjust them if your paths or parameters differ.

In [ ]:
import os
import sys
import io
from contextlib import contextmanager
import numpy as np
import pandas as pd

# Add the project source to the Python path so the cluster utilities are importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path.append(os.path.join(PROJECT_ROOT, 'source_code'))

from clusters.module_clustering_confirmatory_utils import (
    setup_r_environment,
    run_module_quantile_regression_pipeline,
    plot_cluster_feature_brainmaps,
)

# ── paths ─────────────────────────────────────────────────────────────────────
DEPRESSION_DIR  = os.path.join(PROJECT_ROOT, 'data/UKB/F32_notask_STRCO_RSSCHA_RSTIA')
DATA_CSV        = os.path.join(PROJECT_ROOT, 'data/UKB/cohorts/module_connectivity_features_with_covariates.csv')
COLNAME_MAP_CSV = os.path.join(PROJECT_ROOT, 'data/UKB/cohorts/module_connectivity_with_covariates_colname_map.csv')
COMMUNITY_LABELS_PATH = os.path.join(PROJECT_ROOT, 'notebooks/CM.txt')
ATLAS_FILE      = os.path.join(PROJECT_ROOT, 'atlases/deterministic/self_integrated_gpt5_gemini_version/Schaefer1000_TianS4_combined.nii.gz')
PLOTS_DIR       = os.path.join(PROJECT_ROOT, 'reports/plots/schaefer1000+tian54')
FIGURES_DIR     = os.path.join(PROJECT_ROOT, 'reports/figures/schaefer1000+tian54')

# ── analysis settings ─────────────────────────────────────────────────────────
CONNECTIVITY_TYPES = ('functional', 'structural', 'sfc')
DIRECTION_TYPES    = ('internal', 'external')

# ICD-10 comorbidity columns used as model covariates
# I10  = hypertension; Z864 = personal history of psychoactive substance abuse;
# F419 = anxiety disorder, unspecified
ICD_10_COVARIATES = ['I10', 'Z864', 'F419']

# Head-motion covariate columns
fMRI_MOTION_METRIC = 'p24441_i2'  # framewise displacement (fMRI)
dMRI_MOTION_METRIC = 'p24453_i2'  # mean relative head motion (dMRI)

# Cluster column produced by the exploratory module_clustering_main.py
CLUSTER_COL = 'sfc_external_cluster'

# Bootstrap replications for quantile-regression SEs.
# 1 000 is a reasonable development default; raise to 10 000 for final results.
QUANTILE_REGRESSION_BOOTSTRAP_R = 5000

# ── data loading ──────────────────────────────────────────────────────────────
DATA = pd.read_csv(DATA_CSV)
print(f"Data loaded: {DATA.shape[0]} subjects × {DATA.shape[1]} columns")
print(f"Cluster distribution:\n{DATA[CLUSTER_COL].value_counts(dropna=False).to_string()}")

# Community module labels (one integer per ROI, matching atlas ROI ordering)
CM   = np.loadtxt(COMMUNITY_LABELS_PATH)
mods = np.unique(CM)
print(f"\nCommunity labels loaded: {len(CM)} ROIs, {len(mods)} unique modules → {mods}")

# ── column-name mapping ───────────────────────────────────────────────────────
# Maps original feature names (e.g. '1.0_internal_functional') to R-safe
# sanitised names (e.g. 'X1_0_internal_functional').
# Falls back gracefully if the CSV is absent.
colname_map: dict = {}
if os.path.exists(COLNAME_MAP_CSV):
    cm_df = pd.read_csv(COLNAME_MAP_CSV)
    if {'original', 'sanitized'}.issubset(cm_df.columns):
        colname_map = dict(zip(cm_df['original'], cm_df['sanitized']))
        print(f"\nColumn-name map loaded: {len(colname_map)} entries")
    else:
        print("WARNING: colname_map CSV missing 'original'/'sanitized' columns — proceeding without mapping")
else:
    print("INFO: no colname_map CSV found — proceeding without column renaming")

# ── output directories ────────────────────────────────────────────────────────
os.makedirs(PLOTS_DIR,   exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"\nOutput directories ready:\n  Plots   → {PLOTS_DIR}\n  Figures → {FIGURES_DIR}")

## Step 2 — Output capture helpers

`Tee` duplicates every `print()` call to both the notebook cell output and an in-memory buffer. `capture_stdout_to_log` is a context manager that wraps the whole pipeline call and flushes the buffer to a text file when the block exits. This produces a single plain-text log of all Python and R console output.

In [ ]:
class Tee(io.TextIOBase):
    """Write to both the original stdout stream and an in-memory buffer simultaneously."""

    def __init__(self, original, buffer):
        self.original = original
        self.buffer   = buffer

    def write(self, s):
        self.original.write(s)
        self.buffer.write(s)
        return len(s)

    def flush(self):
        self.original.flush()
        self.buffer.flush()


@contextmanager
def capture_stdout_to_log(log_path):
    """Context manager: mirror stdout to *log_path* while keeping notebook output live."""
    original_stdout = sys.stdout
    buffer          = io.StringIO()
    sys.stdout      = Tee(original_stdout, buffer)
    try:
        yield buffer
    finally:
        sys.stdout = original_stdout
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, 'w') as f:
            f.write(buffer.getvalue())
        print(f"[log] stdout captured → {log_path}")

print("Output capture helpers defined.")

## Step 3 — R environment initialisation

`setup_r_environment()` calls `rpy2` to initialise the R process, loads the `quantreg` and `multcomp` packages (installing them if missing), and activates the `pandas ↔ R dataframe` converter. This step must succeed before any quantile regression can run.

**What it checks / does internally:**
- Activates `pandas2ri` auto-conversion so Python DataFrames are passed to R without manual conversion.
- Imports `quantreg` (for `rq`, `summary.rq`, `boot.rq`) and `multcomp`.
- Prints a confirmation message if the packages are available.

In [ ]:
print("[STEP 3] Initialising R environment via rpy2 ...")
setup_r_environment()
print("R environment ready.")

## Step 4 — Quantile-regression pipeline

`run_module_quantile_regression_pipeline()` is the core statistical step. It runs the full analysis for **all modules × all directions × all connectivity types** and returns a `brainmap_significance` dict that maps `(conn_type, dir_type, module) → cluster0_vs_cluster1 FDR-corrected p-value`.

**Internal workflow of the pipeline:**

1. For each `conn_type` in `CONNECTIVITY_TYPES`:
   - Determine the appropriate motion covariates (`p24441_i2` for fMRI, `p24453_i2` for dMRI, both for SFC).
   - For each `dir_type` in `DIRECTION_TYPES`:
     - Open a FDR log file (`modular_{conn}_{dir}_connectivity_FDR_sfc_external.txt`).
     - For each module in `mods`:
       - Call `run_quantile_regression(merged_df, ...)` which:
         - Writes a temporary CSV to `/tmp/` for R to read.
         - Runs three quantile regression models in R (`tau = 0.5`, `se = "boot"`, `R = QUANTILE_REGRESSION_BOOTSTRAP_R`):
           - **Model 1**: `Connectivity ~ Depression + age + sex + motion + comorbidities` — tests *depression vs control*.
           - **Model 2**: `Connectivity ~ Cluster + age + sex + motion + comorbidities` (reference = Control) — tests *Cluster 0 vs Control* and *Cluster 1 vs Control*.
           - **Model 3**: `Connectivity ~ Cluster + age + sex + motion + comorbidities` (reference = Cluster 1) — tests *Cluster 0 vs Cluster 1*.
         - Extracts four p-values and returns them as a dict.
     - Collects all raw p-values across modules and applies **Benjamini–Hochberg FDR** correction in one family.
     - Stores FDR-corrected p-values in `significance_payload[(conn_type, dir_type, module)]`.
   - Calls `plot_module_violin_across_clusters(...)` for this `conn_type` to save violin figures annotated with significance brackets.
2. Returns `brainmap_significance` for downstream brainmap filtering.

**Covariates included in all models:** age (median-centred), sex (`p31`), modality-specific head motion (median-centred), ICD-10 comorbidities (binary factors: hypertension, psychoactive substance history, anxiety).

**Statistical note:** R reports p-values below the floating-point minimum (`2.2e-16`) as exact 0; these are clamped to `2.2e-16` to avoid downstream issues.

**Violin plots** are saved automatically inside this call for each (connectivity type, direction) as:
```
PLOTS_DIR/F32_{conn}_{dir}_module_violin_by_cluster_sfc_external.svg
```
Each figure contains one panel per module, each showing four violin distributions (Control, Depression, Cluster 0, Cluster 1) min-max scaled to [0, 1], annotated with `*`/`**`/`***`/`ns` significance brackets.

In [ ]:
log_path = os.path.join(DEPRESSION_DIR, 'module_connectivity_output_sfc_external.txt')

print("[STEP 4] Running quantile-regression pipeline ...")
print(f"  Modules      : {len(mods)}")
print(f"  Modalities   : {CONNECTIVITY_TYPES}")
print(f"  Directions   : {DIRECTION_TYPES}")
print(f"  Bootstrap R  : {QUANTILE_REGRESSION_BOOTSTRAP_R}")
print(f"  Cluster col  : {CLUSTER_COL}")

with capture_stdout_to_log(log_path):
    brainmap_significance = run_module_quantile_regression_pipeline(
        merged=DATA,                                 # sanitised-name DataFrame (same as final_df here)
        final_df=DATA,                               # original-name DataFrame for plotting
        mods=mods,                                   # unique module IDs from CM.txt
        cluster_col=CLUSTER_COL,                     # 'sfc_external_cluster'
        R=QUANTILE_REGRESSION_BOOTSTRAP_R,           # bootstrap iterations for R SEs
        conn_types=CONNECTIVITY_TYPES,               # ('functional', 'structural', 'sfc')
        dir_types=DIRECTION_TYPES,                   # ('internal', 'external')
        icd_covariates=ICD_10_COVARIATES,            # comorbidity covariates
        fMRI_MOTION_METRIC=fMRI_MOTION_METRIC,       # fMRI head-motion column
        dMRI_MOTION_METRIC=dMRI_MOTION_METRIC,       # dMRI head-motion column
        plots_dir=PLOTS_DIR,                         # destination for violin SVGs
        depressed_subjects_dir=DEPRESSION_DIR,       # destination for FDR log files
        colname_map=colname_map,                     # original → sanitised name mapping
    )

print(f"\nPipeline complete.")
print(f"brainmap_significance entries: {len(brainmap_significance)}")
n_sig = sum(1 for v in brainmap_significance.values() if v is not None and not (isinstance(v, float) and v != v) and v <= 0.05)
print(f"Significant cluster0_vs_cluster1 (FDR q ≤ 0.05): {n_sig} / {len(brainmap_significance)}")

## Step 5 — Inspect significance results

Preview the FDR-corrected significance map produced by the pipeline. Each entry maps `(conn_type, dir_type, module)` to the **cluster0_vs_cluster1 FDR-corrected p-value** used for brainmap filtering. Entries with `q ≤ 0.05` will appear in the brainmaps.

In [ ]:
import math

rows = []
for (conn, direction, module), pval in brainmap_significance.items():
    rows.append({
        'conn_type'  : conn,
        'direction'  : direction,
        'module'     : module,
        'q_cluster0_vs_cluster1': pval,
        'significant': (
            not (isinstance(pval, float) and math.isnan(pval))
            and pval is not None
            and pval <= 0.05
        ),
    })

sig_df = pd.DataFrame(rows).sort_values(['conn_type', 'direction', 'significant'], ascending=[True, True, False])
print(f"Total entries: {len(sig_df)}")
print(f"Significant (q ≤ 0.05): {sig_df['significant'].sum()}")
sig_df.head(40)

## Step 6 — Brain map visualisation

`plot_cluster_feature_brainmaps()` projects module-level connectivity profiles onto the Schaefer 1000 + TianS4 atlas to create spatial brain maps.

**What it produces for each (connectivity type, direction):**

1. **Side-by-side Cluster 0 / Cluster 1 map** — median feature value per module projected onto the parcellation; both maps share the same colour scale (min-max scaled to [0, 1], `YlOrBr` palette). Only modules that pass `pvalue_threshold = 0.05` on the `cluster0_vs_cluster1` FDR-corrected test are included.
2. **Difference map (Cluster 0 − Cluster 1)** — diverging `RdBu_r` palette, symmetric around zero. Shows which modules are more strongly expressed in Cluster 0 (red) vs Cluster 1 (blue).
3. **Optional NIfTI masks** — saved alongside each SVG if `save_niftis=True`.

All maps use fixed cut coordinates (centre of mass of the atlas non-zero voxels) to keep anatomical slices consistent across figures.

**Saved to:** `FIGURES_DIR/{conn}_con/`

In [ ]:
print("[STEP 6] Generating brainmap visualisations ...")
print(f"  Atlas        : {ATLAS_FILE}")
print(f"  Community labels shape: {CM.shape}")
print(f"  Modules passed to brainmaps: {len(mods)}")
print(f"  Significance threshold (q): 0.05")

plot_cluster_feature_brainmaps(
    final_df=DATA,                          # subject-level data with cluster labels
    feature_df=DATA,                        # same DataFrame — also holds features
    cluster_col=CLUSTER_COL,               # 'sfc_external_cluster'
    atlas_img_path=ATLAS_FILE,             # parcellation NIfTI
    community_labels=CM,                   # ROI → module label mapping
    fig_dir=FIGURES_DIR,                   # output root for brain-map SVGs
    modules=mods,                          # module IDs (from CM.txt)
    minmax=True,                           # global min-max scaling before computing medians
    significance_map=brainmap_significance, # cluster0_vs_cluster1 FDR p-values
    pvalue_threshold=0.05,                 # only show significant modules
    save_niftis=True,                      # also save volumetric NIfTI masks
)

print("Brainmap visualisation complete.")

## Step 7 — Output summary

Verify and list all files saved by the pipeline: the full stdout log, per-family FDR correction logs, violin-plot SVGs, and brainmap SVGs/NIfTIs.

In [ ]:
from pathlib import Path

def _list_files(root, pattern, label):
    files = sorted(Path(root).rglob(pattern))
    print(f"\n── {label} ({len(files)} file{'s' if len(files) != 1 else ''}) ──")
    for f in files:
        size_kb = f.stat().st_size / 1024 if f.exists() else 0
        print(f"  {f.relative_to(REPO_ROOT)}  ({size_kb:.1f} KB)")
    return files

print("=" * 70)
print("OUTPUT SUMMARY")
print("=" * 70)

# ── stdout / R output log ─────────────────────────────────────────────────────
log_file = Path(log_path)
if log_file.exists():
    print(f"\n── Stdout log ({log_file.stat().st_size/1024:.1f} KB) ──")
    print(f"  {log_file.relative_to(REPO_ROOT)}")
else:
    print(f"\n[WARNING] Log file not found: {log_path}")

# ── FDR correction logs ───────────────────────────────────────────────────────
_list_files(DEPRESSION_DIR, '*FDR_sfc_external.txt', 'FDR correction logs')

# ── violin plots ──────────────────────────────────────────────────────────────
_list_files(PLOTS_DIR, '*sfc_external*.svg', 'Violin plot SVGs')

# ── brainmap SVGs ─────────────────────────────────────────────────────────────
_list_files(FIGURES_DIR, '*.svg', 'Brainmap SVGs')

# ── NIfTI masks ───────────────────────────────────────────────────────────────
_list_files(FIGURES_DIR, '*.nii.gz', 'NIfTI masks')

print("\n" + "=" * 70)
print("Analysis complete.")